# Gateway Monitoring & A/B Testing: Customer Churn

This notebook demonstrates how to run an A/B test between two ML models using Snowflake's Gateway Monitoring feature. You will:

1. **Deploy both models as inference services** with auto-capture enabled (logs every request/response)
2. **Create a traffic-split gateway** that routes 50/50 live traffic to both services
3. **Send inference traffic** through the gateway via REST API
4. **Create a gateway model monitor** that computes drift and performance metrics across both services
5. **Query and compare metrics** to decide whether the challenger outperforms the baseline

### Prerequisites

Run `00_setup.ipynb` first to create the infrastructure, generate data, train models, and register them in the Model Registry.

### How Gateway A/B Testing Works

```
Client (REST API)
       |
       v
  CHURN_GATEWAY (50/50 traffic split)
       |                |
       v                v
  CHURN_V1_SVC     CHURN_V2_SVC
  (LogReg)         (XGBoost)
       |                |
       v                v
    Auto-Capture Inference Logs
               |
               v
      Gateway Model Monitor
      (drift + performance metrics)
               ^
               |
       Ground Truth Table
       (late-arriving actual labels)
```

### What the Monitor Tracks

The gateway model monitor computes two categories of metrics:

**Drift metrics** (no ground truth needed) -- compare prediction distributions *between services*:
- Population Stability Index (PSI): Are V1 and V2's output distributions diverging?
- Jensen-Shannon Distance: Statistical distance between prediction distributions
- Difference of Means: Systematic shift in average predictions
- Wasserstein Distance: Earth mover's distance between distributions

**Performance metrics** (requires ground truth) -- compare predictions to actual outcomes *per service*:
- Classification Accuracy, F1 Score, Precision, Recall, ROC-AUC

Ground truth is **optional**. Without it, you still get drift metrics. With it, you also get performance metrics by joining predictions to actual labels via a shared ID column.

### Important: Aggregation Window

The minimum aggregation window for gateway monitors is **1 hour**. This means:
- Metrics are computed in 1-hour buckets
- Charts in Snowsight show one data point per hour
- You need inference data spanning multiple hours to see trend lines in the charts
- The Metrics Overview table shows the latest values immediately

## 1. Setup and Connect

Import libraries and establish a Snowflake session.

**Snowsight (Workspaces):** The session is pre-authenticated -- no configuration needed.

**Local development:** Falls back to keypair authentication. Update the `LOCAL CONFIG` section below. The private key is also used later to generate JWT tokens for REST API calls to the gateway.

In [ ]:
import base64
import hashlib
import numpy as np
import pandas as pd
import json
import requests
import time
import uuid
import jwt
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry

In [ ]:
private_key = None  # Set in local dev path; used for JWT auth later

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from cryptography.hazmat.primitives import serialization
    from pathlib import Path

    # ── LOCAL CONFIG (update these for your environment) ──
    ACCOUNT = "SFSENORTHAMERICA-TBSMITH-AWS1"
    USER = "TRASMITH"
    ROLE = "ACCOUNTADMIN"
    KEY_PATH = Path.home() / ".snowflake" / "keys" / "rsa_key.p8"
    # ──────────────────────────────────────────────────────

    with open(KEY_PATH, "rb") as f:
        private_key = serialization.load_pem_private_key(f.read(), password=None)

    private_key_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )

    session = Session.builder.configs({
        "account": ACCOUNT,
        "user": USER,
        "private_key": private_key_bytes,
        "role": ROLE,
        "database": "ML_DEMO",
        "schema": "ML_CHURN",
        "warehouse": "ML_CHURN_WH"
    }).create()

print(f"Connected as: {session.get_current_role()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

## 2. Retrieve Registered Models and Test Data

The models were trained and registered in `00_setup.ipynb`. Here we retrieve them from the Model Registry and load the test data from Snowflake.

In [ ]:
reg = Registry(session=session)
model = reg.get_model("CHURN_MODEL")
mv_v1 = model.version("V1")
mv_v2 = model.version("V2")
print(f"Retrieved CHURN_MODEL V1 and V2 from registry")

FEATURE_COLS = ["TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES",
                "CONTRACT_TYPE", "NUM_SUPPORT_TICKETS", "INTERNET_SERVICE"]

test_data = session.table("ML_DEMO.ML_CHURN.CHURN_TEST").to_pandas()
X_test = test_data[FEATURE_COLS]
y_test = test_data["CHURNED"]
print(f"Loaded test data: {len(test_data)} rows")

## 3. Deploy Inference Services with Auto-Capture

Each model version is deployed as a separate inference service on SPCS (Snowpark Container Services).

**Auto-Capture** (`autocapture=True`) automatically logs every request and response processed by the service. This is what feeds data into the gateway model monitor later. Auto-capture logs include:
- Request payload (input features)
- Response payload (predictions)
- Model version and service identifiers
- Gateway routing metadata
- Timestamps and HTTP status codes
- Extra columns (like `request_id`) for ground truth join

**Note:** Auto-capture is immutable -- you cannot enable/disable it on an existing service. You must set it at service creation time. First deploy may take up to 10 minutes while the container image is built.

In [ ]:
# Deploy v1 service
# Change COMPUTE_POOL to match your account's available compute pool
COMPUTE_POOL = "ML_ONLINE_CPU_POOL"

mv_v1.create_service(
    service_name="CHURN_V1_SVC",
    service_compute_pool=COMPUTE_POOL,
    ingress_enabled=True,
    autocapture=True
)
print("Deploying CHURN_V1_SVC... (may take up to 10 minutes)")

In [ ]:
# Deploy v2 service
mv_v2.create_service(
    service_name="CHURN_V2_SVC",
    service_compute_pool=COMPUTE_POOL,
    ingress_enabled=True,
    autocapture=True
)
print("Deploying CHURN_V2_SVC... (may take up to 10 minutes)")

In [ ]:
# Wait for both services to become READY
def wait_for_service(session, service_name, timeout=900):
    start = time.time()
    while time.time() - start < timeout:
        result = session.sql(f"SHOW ENDPOINTS IN SERVICE ML_DEMO.ML_CHURN.{service_name}").collect()
        for row in result:
            row_dict = row.as_dict()
            if row_dict.get("ingress_url") and row_dict["ingress_url"] != "Endpoints provisioning in progress...":
                print(f"\n{service_name}: READY (endpoint: {row_dict['ingress_url']})")
                return row_dict["ingress_url"]
        elapsed = int(time.time() - start)
        print(f"  {service_name}: provisioning... ({elapsed}s)", end="\r")
        time.sleep(30)
    print(f"\n{service_name}: TIMEOUT after {timeout}s")
    return None

print("Waiting for services to become ready...")
ep_v1 = wait_for_service(session, "CHURN_V1_SVC")
ep_v2 = wait_for_service(session, "CHURN_V2_SVC")
print(f"\nV1 endpoint: {ep_v1}")
print(f"V2 endpoint: {ep_v2}")

## 4. Create Traffic-Split Gateway

The Snowflake Gateway provides a **stable URL** that routes traffic to multiple inference services. Unlike individual service endpoints (which change when a service is recreated), the gateway URL is permanent.

Key capabilities:
- **Traffic splitting**: Route requests to services by percentage (e.g., 50/50, 90/10 canary)
- **Automatic failover**: If a service goes down, traffic is redirected to healthy targets
- **Schema evolution**: Send a superset of features -- each model ignores fields it doesn't need

The gateway specification requires:
- `type: traffic_split` and `split_type: custom`
- Targets as fully qualified `DB.SCHEMA.SERVICE!endpoint` references
- Weights that sum to exactly 100

You can change the traffic split at any time with `ALTER GATEWAY` -- no need to recreate it.

In [ ]:
# Create 50/50 traffic split gateway
session.sql("""
CREATE OR REPLACE GATEWAY ML_DEMO.ML_CHURN.CHURN_GATEWAY
  FROM SPECIFICATION $$
spec:
  type: traffic_split
  split_type: custom
  targets:
  - type: endpoint
    value: ML_DEMO.ML_CHURN.CHURN_V1_SVC!inference
    weight: 50
  - type: endpoint
    value: ML_DEMO.ML_CHURN.CHURN_V2_SVC!inference
    weight: 50
$$;
""").collect()
print("Gateway CHURN_GATEWAY created with 50/50 traffic split")

In [ ]:
# Get gateway endpoint URL (wait for provisioning to complete)
gw_endpoint = None
timeout = 300  # 5 minutes
start = time.time()

while time.time() - start < timeout:
    gw_desc = session.sql("DESC GATEWAY ML_DEMO.ML_CHURN.CHURN_GATEWAY").collect()
    for row in gw_desc:
        row_dict = row.as_dict()
        url = row_dict.get("ingress_url", "")
        if url and "snowflakecomputing" in url:
            gw_endpoint = url
            break
    if gw_endpoint:
        break
    elapsed = int(time.time() - start)
    print(f"  Gateway provisioning... ({elapsed}s)", end="\r")
    time.sleep(10)

if gw_endpoint:
    GATEWAY_URL = f"https://{gw_endpoint}/predict"
    print(f"\nGateway inference URL: {GATEWAY_URL}")
else:
    print("\nGateway endpoint not ready within timeout.")
    print("Check: DESC GATEWAY ML_DEMO.ML_CHURN.CHURN_GATEWAY")
    print("Then set manually: GATEWAY_URL = 'https://<ingress_url>/predict'")

## 5. Run Inference Loop (~5 minutes)

This sends inference requests through the gateway's REST endpoint. The gateway routes each request to V1 or V2 based on the 50/50 split. Both services have auto-capture enabled, so every request/response is logged automatically.

**`extra_columns`**: We include a `request_id` field in each request payload and declare it in `extra_columns`. This field is NOT sent to the model -- it's captured in the inference logs under `RECORD_ATTRIBUTES:"snow.model_serving.request.extra_columns.request_id"`. This is the join key used later to match predictions with ground truth labels.

**Important**: Column names in `extra_columns` must be **lowercase** to match the ground truth table columns. Snowflake preserves the exact casing from the REST payload.

**Authentication**: The cell below auto-detects the environment:
- **Local development**: Generates a keypair JWT from the same `rsa_key.p8` used for the Snowpark session. Auto-refreshes every 55 minutes.
- **Snowsight (Workspaces)**: Uses the session's built-in token. You may need to set a PAT token if the session token doesn't have REST endpoint access.

In [ ]:
JWT_LIFETIME = 3600  # 1 hour

if private_key is not None:
    # Local dev: generate JWT from keypair
    def generate_jwt_token():
        pub_bytes = private_key.public_key().public_bytes(
            serialization.Encoding.DER, serialization.PublicFormat.SubjectPublicKeyInfo
        )
        fingerprint = "SHA256:" + base64.b64encode(hashlib.sha256(pub_bytes).digest()).decode()
        qualified = f"{ACCOUNT.upper()}.{USER.upper()}"
        now = int(time.time())
        payload = {
            "iss": f"{qualified}.{fingerprint}",
            "sub": qualified,
            "iat": now,
            "exp": now + JWT_LIFETIME,
        }
        return jwt.encode(payload, private_key, algorithm="RS256")

    token = generate_jwt_token()
    print("JWT token generated from keypair")
else:
    # Snowsight: use session token or set a PAT below
    token = session.connection.rest.token
    generate_jwt_token = None
    print("Using session token for REST API calls")

HEADERS = {
    "Authorization": f'Snowflake Token="{token}"',
    "Content-Type": "application/json"
}

In [ ]:
# Run inference loop for ~5 minutes
assert "snowflakecomputing" in GATEWAY_URL, f"GATEWAY_URL not valid: {GATEWAY_URL}"

DURATION_SECONDS = 300  # 5 minutes
BATCH_SIZE = 5
SLEEP_BETWEEN = 0.5
TOKEN_REFRESH_INTERVAL = 3300  # refresh token every 55 minutes

all_request_ids = []
all_actual_labels = []
batch_count = 0
error_count = 0
last_token_time = time.time()

X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

start_time = time.time()
print(f"Starting inference loop for {DURATION_SECONDS}s...")
print(f"Target: {GATEWAY_URL}")

while time.time() - start_time < DURATION_SECONDS:
    # Refresh token if needed (for long-running loops)
    if generate_jwt_token is not None and time.time() - last_token_time > TOKEN_REFRESH_INTERVAL:
        token = generate_jwt_token()
        HEADERS["Authorization"] = f'Snowflake Token="{token}"'
        last_token_time = time.time()
        print("  Token refreshed")

    indices = np.random.choice(len(X_test_reset), size=BATCH_SIZE, replace=False)
    sample = X_test_reset.iloc[indices].copy()
    actual_labels = y_test_reset.iloc[indices].tolist()

    request_ids = [str(uuid.uuid4()) for _ in range(BATCH_SIZE)]
    sample["request_id"] = request_ids

    payload = {
        "dataframe_records": json.loads(sample.to_json(orient="records")),
        "extra_columns": ["request_id"]
    }

    try:
        resp = requests.post(GATEWAY_URL, headers=HEADERS, json=payload, timeout=30)
        if resp.status_code == 200:
            all_request_ids.extend(request_ids)
            all_actual_labels.extend(actual_labels)
            batch_count += 1
        else:
            error_count += 1
            if error_count <= 3:
                print(f"Error {resp.status_code}: {resp.text[:200]}")
    except Exception as e:
        error_count += 1
        if error_count <= 3:
            print(f"Request error: {e}")

    if batch_count % 60 == 0 and batch_count > 0:
        elapsed = int(time.time() - start_time)
        print(f"  {elapsed}s elapsed | {batch_count} batches sent | {error_count} errors")

    time.sleep(SLEEP_BETWEEN)

elapsed = int(time.time() - start_time)
print(f"\nDone! {batch_count} batches ({batch_count * BATCH_SIZE} predictions) in {elapsed}s")
print(f"Errors: {error_count}")
print(f"Request IDs collected: {len(all_request_ids)}")

## 6. Verify Auto-Captured Inference Logs

Query the `INFERENCE_TABLE()` function to confirm that auto-capture is logging requests from both model versions. The first query shows recent log entries with model version, function name, gateway hop ID, and request ID. The second query counts requests per version to verify the 50/50 traffic split is working.

In [ ]:
# Query inference table to verify auto-capture is working
inference_logs = session.sql("""
SELECT 
    RESOURCE_ATTRIBUTES:"snow.model.version.name"::VARCHAR AS model_version,
    RECORD_ATTRIBUTES:"snow.model_serving.function.name"::VARCHAR AS function_name,
    RECORD_ATTRIBUTES:"snow.model_serving.last_hop_id"::VARCHAR AS gateway_hop,
    RECORD_ATTRIBUTES:"snow.model_serving.request.extra_columns.request_id"::VARCHAR AS request_id,
    TIMESTAMP
FROM TABLE(INFERENCE_TABLE('CHURN_MODEL'))
ORDER BY TIMESTAMP DESC
LIMIT 10;
""").to_pandas()

print("Recent inference logs (should show requests from both V1 and V2):")
inference_logs

In [ ]:
# Verify traffic split: count requests per model version
traffic_split = session.sql("""
SELECT 
    RESOURCE_ATTRIBUTES:"snow.model.version.name"::VARCHAR AS model_version,
    COUNT(*) AS request_count
FROM TABLE(INFERENCE_TABLE('CHURN_MODEL'))
GROUP BY 1
ORDER BY 1;
""").to_pandas()

print("Traffic distribution across model versions:")
print(traffic_split)
print("\nExpected: ~50/50 split between V1 and V2")

## 7. Populate Ground Truth Table

Ground truth labels are **optional** for the gateway model monitor. Without them, you still get **drift metrics** (PSI, Jensen-Shannon, etc.) that compare prediction distributions between services. With ground truth, you additionally get **performance metrics** (accuracy, F1, precision, recall, ROC-AUC).

In production, ground truth labels typically arrive late -- days or weeks after predictions are made. For example, you predict churn today but only know if the customer actually churned 30 days later. The monitor handles this by joining late-arriving labels to auto-captured inference logs on the `request_id` column whenever it refreshes.

Here we simulate late-arriving labels by writing the actual test labels for the requests we just sent.

**Important**: The ground truth table must use **quoted lowercase** column names (e.g., `"request_id"`, `"churned"`) to match the `extra_columns` field names from the REST payload. Snowflake's default behavior uppercases unquoted identifiers, which would break the join.

In [ ]:
# Write ground truth labels
# Column names MUST be lowercase (quoted) to match the extra_columns field names
# from the REST payload. The gateway monitor joins on these exact column names.
ground_truth_df = pd.DataFrame({
    "request_id": all_request_ids,
    "churned": all_actual_labels
})

print(f"Writing {len(ground_truth_df)} ground truth labels...")

# Recreate the table with quoted lowercase columns
session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.GROUND_TRUTH").collect()
session.sql("""
CREATE TABLE ML_DEMO.ML_CHURN.GROUND_TRUTH (
    "request_id" VARCHAR,
    "churned" NUMBER
)
""").collect()

session.write_pandas(
    ground_truth_df,
    table_name="GROUND_TRUTH",
    database="ML_DEMO",
    schema="ML_CHURN",
    overwrite=False,
    quote_identifiers=True
)
print("Ground truth table populated.")

# Verify
count = session.sql("SELECT COUNT(*) AS CNT FROM ML_DEMO.ML_CHURN.GROUND_TRUTH").collect()
print(f"Rows in GROUND_TRUTH: {count[0]['CNT']}")

## 8. Create Gateway Model Monitor

The gateway model monitor is a single object that tracks **all services behind the gateway**. You do not create a separate monitor per service. The monitor:

1. Wakes up every `REFRESH_INTERVAL` (1 minute) to check for new inference data
2. Reads auto-captured inference logs from `INFERENCE_TABLE`
3. Joins predictions to actual labels in `GROUND_TRUTH` via `ID_COLUMNS` (request_id)
4. Aggregates metrics into `AGGREGATION_WINDOW` (1 hour) buckets
5. Exposes results via `MODEL_MONITOR_*_METRIC()` SQL functions and the Snowsight Monitoring tab

**Key parameters:**
- `GATEWAY`: Makes this a gateway monitor (vs `VERSION` for a model version monitor)
- `GROUND_TRUTH` + `ID_COLUMNS`: Optional. Omit both for drift-only monitoring
- `ACTUAL_CLASS_COLUMNS`: Which column in the ground truth table has the real labels
- `AGGREGATION_WINDOW`: Minimum 1 hour for gateway monitors

The monitor **cannot be reconfigured** after creation (e.g., you can't add ground truth later). To change the configuration, drop and recreate it.

**All metrics are computed automatically** -- you don't select which metrics to track. You choose which to query at read time via the `METRIC_NAME` parameter.

In [ ]:
# Create the gateway model monitor
session.sql("""
CREATE OR REPLACE MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_AB_MONITOR WITH
    MODEL = ML_DEMO.ML_CHURN.CHURN_MODEL
    GATEWAY = ML_DEMO.ML_CHURN.CHURN_GATEWAY
    FUNCTION = 'predict'
    WAREHOUSE = ML_CHURN_WH
    REFRESH_INTERVAL = '1 minute'
    AGGREGATION_WINDOW = '1 hour'
    GROUND_TRUTH = ML_DEMO.ML_CHURN.GROUND_TRUTH
    ID_COLUMNS = ('request_id')
    ACTUAL_CLASS_COLUMNS = ('churned')
    COMMENT = 'A/B test: LogisticRegression (V1) vs XGBoost (V2)'
""").collect()

print("Gateway model monitor CHURN_AB_MONITOR created successfully!")

In [ ]:
# Verify monitor status
monitor_desc = session.sql("""
DESCRIBE MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_AB_MONITOR
""").to_pandas()

print("Monitor description:")
monitor_desc

## 9. Query Monitoring Metrics

You can query metrics programmatically using three SQL functions:

| Function | What it measures | Requires ground truth? |
|----------|-----------------|----------------------|
| `MODEL_MONITOR_DRIFT_METRIC` | Prediction distribution differences **between services** (PSI, Jensen-Shannon, Wasserstein, Difference of Means) | No |
| `MODEL_MONITOR_PERFORMANCE_METRIC` | Model accuracy **per service** (Accuracy, F1, Precision, Recall, ROC-AUC) | Yes |
| `MODEL_MONITOR_STAT_METRIC` | Record counts and null rates | No |

**Drift metrics** require `SERVICE`, `BASE_SERVICE` (the baseline to compare against), and `COLUMN_NAME` (the prediction column, quoted lowercase: `'"output_feature_0"'`).

**Performance metrics** require `SERVICE` and optionally `GRANULARITY` (e.g., `'1 HOUR'`).

**Snowsight**: Navigate to **AI & ML > Models > Gateways > CHURN_GATEWAY > Monitoring** to see the dashboard. The Metrics Overview table shows the latest values per service. Charts show time-series data -- you need data spanning multiple hours to see trend lines.

**Note**: Since the aggregation window is 1 hour, metrics appear once a complete hour-bucket of data has been processed. If you just ran the inference loop, the data may fall in the current (incomplete) hour. Change the Snowsight time range to "Last 3 hours" or "Last 24 hours" to see data from completed buckets.

In [ ]:
# Drift metric: compare V2 predictions to V1 baseline (Population Stability Index)
try:
    drift_results = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_DRIFT_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_AB_MONITOR',
            METRIC_NAME => 'POPULATION_STABILITY_INDEX',
            COLUMN_NAME => '"output_feature_0"',
            SERVICE => 'ML_DEMO.ML_CHURN.CHURN_V2_SVC',
            BASE_SERVICE => 'ML_DEMO.ML_CHURN.CHURN_V1_SVC',
            GRANULARITY => '1 HOUR'
        )
    );
    """).to_pandas()
    print("Drift metrics (PSI - V2 vs V1 baseline):")
    drift_results
except Exception as e:
    print(f"Drift metrics not yet available (need ~1 hour of data): {e}")

In [ ]:
# Performance metric: accuracy for each service
try:
    perf_v1 = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_PERFORMANCE_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_AB_MONITOR',
            METRIC_NAME => 'CLASSIFICATION_ACCURACY',
            SERVICE => 'ML_DEMO.ML_CHURN.CHURN_V1_SVC',
            GRANULARITY => '1 HOUR'
        )
    );
    """).to_pandas()
    print("Performance (Accuracy) - V1 (LogisticRegression):")
    display(perf_v1)
except Exception as e:
    print(f"V1 performance metrics not yet available: {e}")

try:
    perf_v2 = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_PERFORMANCE_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_AB_MONITOR',
            METRIC_NAME => 'CLASSIFICATION_ACCURACY',
            SERVICE => 'ML_DEMO.ML_CHURN.CHURN_V2_SVC',
            GRANULARITY => '1 HOUR'
        )
    );
    """).to_pandas()
    print("\nPerformance (Accuracy) - V2 (XGBoost):")
    display(perf_v2)
except Exception as e:
    print(f"V2 performance metrics not yet available: {e}")

In [ ]:
# F1 score comparison
try:
    f1_v1 = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_PERFORMANCE_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_AB_MONITOR',
            METRIC_NAME => 'F1_SCORE',
            SERVICE => 'ML_DEMO.ML_CHURN.CHURN_V1_SVC',
            GRANULARITY => '1 HOUR'
        )
    );
    """).to_pandas()
    
    f1_v2 = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_PERFORMANCE_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_AB_MONITOR',
            METRIC_NAME => 'F1_SCORE',
            SERVICE => 'ML_DEMO.ML_CHURN.CHURN_V2_SVC',
            GRANULARITY => '1 HOUR'
        )
    );
    """).to_pandas()
    
    print("F1 Score - V1 (LogisticRegression):")
    display(f1_v1[["EVENT_TIMESTAMP", "METRIC_VALUE", "COUNT_USED", "CI_VALUE"]])
    print("\nF1 Score - V2 (XGBoost):")
    display(f1_v2[["EVENT_TIMESTAMP", "METRIC_VALUE", "COUNT_USED", "CI_VALUE"]])
except Exception as e:
    print(f"F1 metrics not yet available: {e}")

## 10. (Optional) Shift Traffic to Winner

Once the monitoring metrics confirm V2 outperforms V1 (higher accuracy, better F1, acceptable drift), you can shift traffic using `ALTER GATEWAY`. Common patterns:

- **Gradual rollout**: 50/50 -> 70/30 -> 90/10 -> 100/0
- **Canary**: Start at 95/5, watch metrics, then increase
- **Instant cutover**: Jump to 0/100

Note: A target with weight 0 still exists in the gateway but receives no traffic. Traffic is never failed over to a 0% endpoint.

In [ ]:
# Uncomment to shift traffic 100% to V2
# session.sql("""
# ALTER GATEWAY ML_DEMO.ML_CHURN.CHURN_GATEWAY
# FROM SPECIFICATION $$
# spec:
#   type: traffic_split
#   split_type: custom
#   targets:
#   - type: endpoint
#     value: ML_DEMO.ML_CHURN.CHURN_V1_SVC!inference
#     weight: 0
#   - type: endpoint
#     value: ML_DEMO.ML_CHURN.CHURN_V2_SVC!inference
#     weight: 100
# $$;
# """).collect()
# print("Traffic shifted 100% to V2 (XGBoost)")

## 11. Cleanup

Run these cells to tear down all resources created by this notebook. This does **not** remove the shared resources from `00_setup` (database, tables, models).

In [ ]:
# Drop monitor
session.sql("DROP MODEL MONITOR IF EXISTS ML_DEMO.ML_CHURN.CHURN_AB_MONITOR").collect()
print("Dropped CHURN_AB_MONITOR")

# Drop gateway
session.sql("DROP GATEWAY IF EXISTS ML_DEMO.ML_CHURN.CHURN_GATEWAY").collect()
print("Dropped CHURN_GATEWAY")

# Delete services
mv_v1.delete_service("CHURN_V1_SVC")
print("Deleted CHURN_V1_SVC")

mv_v2.delete_service("CHURN_V2_SVC")
print("Deleted CHURN_V2_SVC")

print("\nGateway resources cleaned up (models and data remain for other demos).")

In [ ]:
# Optional: full cleanup (removes ALL shared resources -- only run after all demos)
# session.sql("DROP MODEL IF EXISTS ML_DEMO.ML_CHURN.CHURN_MODEL").collect()
# session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.GROUND_TRUTH").collect()
# session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CHURN_TRAIN").collect()
# session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CHURN_TEST").collect()
# session.sql("DROP SCHEMA IF EXISTS ML_DEMO.ML_CHURN").collect()
# session.sql("DROP DATABASE IF EXISTS ML_DEMO").collect()
# session.sql("DROP WAREHOUSE IF EXISTS ML_CHURN_WH").collect()
# print("Full cleanup complete.")

In [ ]:
session.close()
print("Session closed.")